# A.3: The Self-Instruct similarity filter

Keep a generated instruction only if its ROUGE-L similarity to every instruction already in the pool is below 0.7.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 Instruction templates for a labeled dataset
import random, re, hashlib

# From A.2 Exact and near-duplicate detection
def normalize(s):
    return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", s.lower())).strip()


In [ ]:
def lcs(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            dp[i + 1][j + 1] = dp[i][j] + 1 if x == y else max(dp[i][j + 1], dp[i + 1][j])
    return dp[-1][-1]

def rouge_l(a, b):
    a, b = normalize(a).split(), normalize(b).split()
    l = lcs(a, b)
    if l == 0:
        return 0.0
    p, r = l / len(b), l / len(a)
    return 2 * p * r / (p + r)

task_pool = ["Write a short poem about the ocean.", "Explain how a rainbow forms.",
             "List three uses of baking soda."]
candidates = ["Write a short poem about the moon.",
              "Explain how a volcano forms.",
              "Describe the rules of chess in simple terms.",
              "List three uses of vinegar."]
for c in candidates:
    score = max(rouge_l(c, t) for t in task_pool)
    keep = score < 0.7
    print(f"{score:.2f} {'keep' if keep else 'drop'}  {c!r}")
    if keep:
        task_pool.append(c)


**Expected output**

Output:

```text
0.86 drop  'Write a short poem about the moon.'
0.80 drop  'Explain how a volcano forms.'
0.14 keep  'Describe the rules of chess in simple terms.'
0.73 drop  'List three uses of vinegar.'
```
